# Volatility Regression 03 — nested-block next-5m volatility OLS

**Question.** Which groups of current-bar information add out-of-sample linear forecast value for the volatility of the next five-minute bar, beyond the bar's own volatility?

**Design** (VR 007, approved by Ivan on 2026-10-09):

- **Targets:** paired, as in 01. `parkinson` is $\log(H/L)/\sqrt{4\log 2}$ from the next perpetual trade bar; `index_abs` is $|r^{\text{index}}_{t+1}|/\sqrt{2/\pi}$. The response is $Y=\log(\sigma+\epsilon_{\text{train}})$.
- **No development cutoff:** this branch is independent of HSMM.
- **Excluded by design:** option smile shape, fitted smile, wings and spreads, and `atm_iv_d60`. On a 0DTE chain these do not exist in the hours before expiry. L2 instability statistics are also excluded, because they are absent before 2026-10-02.
- **Nested blocks:**
  - B0: returns, volatility and clock;
  - B1: B0 plus flow and basis/OI;
  - B2: B1 plus the L2 time-weighted book;
  - B3: B2 plus ATM IV level and short ATM IV changes.
- **Coverage and budget rules:**
  - A feature is admitted per asset if its coverage on development rows is at least 80%.
  - A fit is attempted only if $n_{\text{train}} \ge 20p$.
  - `log_rv_4h_tol` (at least 44 of 48 index returns, rescaled) replaces the shared strict 48/48 column.
- **Evaluation:**
  - The last 25% of sessions are the test set and are scored once.
  - Earlier sessions are scored by forward chaining: train on sessions $1..k$, validate on session $k+1$.
  - Every block is scored on its **own** complete rows and on rows **common** to all blocks; the common rows are where block increments are comparable.
- **References:** train mean (the benchmark for $R^2_{\text{OOS}}$) and persistence. There is no HAR study.
- **Deletion:** greedy backward deletion inside B2. A removal is accepted only if it lowers the mean fold RMSE **and** helps in a majority of folds. The test set is never used.
- **Collector vintage:** test residuals are reported by data-derived collector vintage, read from `02_vintage_boundaries.csv`. **Run notebook 02 first.**

## 1. Environment and data (DuckDB read-only)

In [1]:
from pathlib import Path
import os, sys, json, subprocess
import numpy as np
import pandas as pd
import duckdb
from dotenv import load_dotenv

HERE = Path.cwd().resolve()
ROOT = next((p for p in [HERE, *HERE.parents] if (p / "src" / "data_engineering").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Launch from the repository root or one of its subfolders.")
sys.path.insert(0, str(ROOT / "src"))
load_dotenv(ROOT / ".env")
DATA_ROOT = os.getenv("DERIBIT_DATA_ROOT")
if not DATA_ROOT:
    raise RuntimeError("DERIBIT_DATA_ROOT must be set in .env.")
DB_PATH = Path(DATA_ROOT) / "database" / "research.duckdb"
if not DB_PATH.is_file():
    raise FileNotFoundError(DB_PATH)

from data_engineering.modeling import vol_ols_5m_blocks as vb
from data_engineering.modeling import vol_ols_5m as base
OUTPUT = ROOT / "notebooks" / "Volatility_Regression"

TEST_FRACTION = 0.25
MIN_TRAIN_SESSIONS = 6
DELETE_BLOCK = "B2_l2_book"
MAX_DROPS = 15

with duckdb.connect(str(DB_PATH), read_only=True) as con:
    con.execute("SET TimeZone = 'UTC'")
    df = con.execute("""
        SELECT * FROM time_series_feature_state
        ORDER BY underlying, state_time
    """).df()
ASSETS = tuple(sorted(df["underlying"].dropna().astype(str).unique()))
vintage_path = OUTPUT / "02_vintage_boundaries.csv"
boundaries = pd.read_csv(vintage_path) if vintage_path.is_file() else None
if boundaries is None:
    print("WARNING: 02_vintage_boundaries.csv not found; vintages will be 'unknown'. Run 02 first.")
print("Blocks version:", vb.VOL_OLS_BLOCKS_VERSION, "| base:", base.VOL_OLS_5M_VERSION)
print("Assets:", ASSETS, "| rows:", len(df))
print("Span:", df.state_time.min(), "to", df.state_time.max())

Blocks version: 0.1.0 | base: 0.1.2
Assets: ('AVAX', 'BTC', 'ETH', 'HYPE', 'SOL', 'TRX', 'XRP') | rows: 38304
Span: 2026-09-19 08:00:00+00:00 to 2026-10-08 07:55:00+00:00


In [2]:
for pattern in ("test_vol_ols_5m.py", "test_vol_ols_5m_blocks.py"):
    subprocess.run([sys.executable, "-m", "unittest", "discover", "-s", str(ROOT / "tests"),
                    "-p", pattern, "-v"], cwd=ROOT, check=True)

## 2. Fit blocks per asset and target

In [3]:
fold_frames, admission_frames, clock_frames, loss_frames = [], [], [], []
deletion_frames, reduced_rows, vintage_rows, plan_rows, errors = [], [], [], [], []

for asset in ASSETS:
    g = df[df["underlying"] == asset]
    try:
        meta, X = vb.panel(g)
        valid = vb.paired_valid(meta)
        plan = vb.session_plan(meta, valid, TEST_FRACTION, MIN_TRAIN_SESSIONS)
        lab = plan["labels"]
        dev_rows = np.isin(lab, plan["dev"]) & valid
        test_rows = np.isin(lab, plan["test"]) & valid
        available, adm = vb.admit_features(X, dev_rows)
        adm["underlying"] = asset
        admission_frames.append(adm)
        vint = vb.assign_vintage(meta["state_time"], boundaries, asset)
        plan_rows.append({"underlying": asset, "sessions": len(plan["sessions"]),
                          "dev_first": plan["dev"][0], "dev_last": plan["dev"][-1],
                          "test_first": plan["test"][0], "test_last": plan["test"][-1],
                          "folds": len(plan["folds"]), "paired_valid_rows": int(valid.sum()),
                          "test_vintages": ",".join(sorted(set(vint[test_rows])))})
        cc = vb.clock_coverage(meta, X, valid, available)
        cc["underlying"] = asset
        clock_frames.append(cc)
        ul = vb.unique_loss(X, dev_rows, vb.nested_features(vb.BLOCK_ORDER[-1], available))
        ul["underlying"] = asset
        loss_frames.append(ul)
    except Exception as exc:
        errors.append({"underlying": asset, "target": "*", "reason": f"{type(exc).__name__}: {exc}"})
        print(f"{asset:>6} NOT PLANNED: {errors[-1]['reason']}")
        continue

    for target in base.TARGETS:
        try:
            res, final = vb.evaluate_blocks(meta, X, valid, plan, target, available)
            res.insert(0, "underlying", asset)
            fold_frames.append(res)

            for (population, block), r in final.items():
                resid = r["y"][r["idx"]] - r["pred"]
                for v in sorted(set(vint[r["idx"]])):
                    m = vint[r["idx"]] == v
                    vintage_rows.append({"underlying": asset, "target": target,
                                         "population": population, "model": block,
                                         "vintage": v, "n": int(m.sum()),
                                         "mean_residual": float(resid[m].mean()),
                                         "rmse": float(np.sqrt(np.mean(resid[m] ** 2)))})

            start = vb.nested_features(DELETE_BLOCK, available)
            keep, hist = vb.backward_by_folds(meta, X, valid, plan, target, start, MAX_DROPS)
            if len(hist):
                hist.insert(0, "underlying", asset)
                deletion_frames.append(hist)
            complete = np.isfinite(X[start].to_numpy(float)).all(axis=1)
            y, pers, _ = vb.target_y(meta, target, dev_rows)
            for label, names in ((DELETE_BLOCK + "_start", start), (DELETE_BLOCK + "_reduced", keep)):
                r = vb.fit_score(X, y, dev_rows & complete, test_rows & complete, names)
                rec = {"underlying": asset, "target": target, "model": label,
                       "p": len(r["names"]), "n_train": r["n_train"], "n_test": r["n_score"],
                       "fitted": r["ok"], "reason": r.get("reason", "")}
                if r["ok"]:
                    ps = base.score(y[r["idx"]], pers[r["idx"]], r["train_mean"], "test")
                    rec.update({"log_rmse": r["log_rmse"], "r2_vs_train_mean": r["r2_vs_train_mean"],
                                "persistence_log_rmse": ps["log_rmse"],
                                "rmse_ratio_vs_persistence": r["log_rmse"] / ps["log_rmse"]})
                reduced_rows.append(rec)
            print(f"{asset:>6} {target:<10} folds={len(plan['folds'])} "
                  f"B2 p={len(start)} -> {len(keep)}")
        except Exception as exc:
            errors.append({"underlying": asset, "target": target,
                           "reason": f"{type(exc).__name__}: {exc}"})
            print(f"{asset:>6} {target:<10} NOT FIT: {errors[-1]['reason']}")

folds_df = pd.concat(fold_frames, ignore_index=True) if fold_frames else pd.DataFrame()
admission_df = pd.concat(admission_frames, ignore_index=True) if admission_frames else pd.DataFrame()
clock_df = pd.concat(clock_frames, ignore_index=True) if clock_frames else pd.DataFrame()
loss_df = pd.concat(loss_frames, ignore_index=True) if loss_frames else pd.DataFrame()
deletion_df = pd.concat(deletion_frames, ignore_index=True) if deletion_frames else pd.DataFrame()
reduced_df = pd.DataFrame(reduced_rows)
vintage_df = pd.DataFrame(vintage_rows)
plan_df = pd.DataFrame(plan_rows)
errors_df = pd.DataFrame(errors)
display(plan_df)
if not errors_df.empty:
    display(errors_df)

  AVAX parkinson  folds=7 B2 p=39 -> 25
  AVAX index_abs  folds=7 B2 p=39 -> 25
   BTC parkinson  folds=7 B2 p=39 -> 26
   BTC index_abs  folds=7 B2 p=39 -> 24
   ETH parkinson  folds=7 B2 p=39 -> 28
   ETH index_abs  folds=7 B2 p=39 -> 24
  HYPE parkinson  folds=7 B2 p=39 -> 24
  HYPE index_abs  folds=7 B2 p=39 -> 24
   SOL parkinson  folds=7 B2 p=39 -> 31
   SOL index_abs  folds=7 B2 p=39 -> 24
   TRX parkinson  NOT FIT: ValueError: Fewer than 3 usable folds for deletion
   TRX index_abs  NOT FIT: ValueError: Fewer than 3 usable folds for deletion
   XRP parkinson  folds=7 B2 p=39 -> 24
   XRP index_abs  folds=7 B2 p=39 -> 24


,underlying,sessions,dev_first,dev_last,test_first,test_last,folds,paired_valid_rows,test_vintages
0,AVAX,17,2026-09-22,2026-10-04,2026-10-05,2026-10-08,7,4026,"eventtime_fix_unstamped,futures_5m_l2_v0.3.0,f..."
1,BTC,17,2026-09-22,2026-10-04,2026-10-05,2026-10-08,7,4607,"eventtime_fix_unstamped,futures_5m_l2_v0.3.0,f..."
2,ETH,17,2026-09-22,2026-10-04,2026-10-05,2026-10-08,7,4606,"eventtime_fix_unstamped,futures_5m_l2_v0.3.0,f..."
3,HYPE,17,2026-09-22,2026-10-04,2026-10-05,2026-10-08,7,3355,"eventtime_fix_unstamped,futures_5m_l2_v0.3.0,f..."
4,SOL,17,2026-09-22,2026-10-04,2026-10-05,2026-10-08,7,4608,"eventtime_fix_unstamped,futures_5m_l2_v0.3.0,f..."
5,TRX,17,2026-09-22,2026-10-04,2026-10-05,2026-10-08,7,784,"futures_5m_l2_v0.3.0,futures_5m_l2_v0.3.1"
6,XRP,17,2026-09-22,2026-10-04,2026-10-05,2026-10-08,7,4100,"eventtime_fix_unstamped,futures_5m_l2_v0.3.0,f..."


,underlying,target,reason
0,TRX,parkinson,ValueError: Fewer than 3 usable folds for dele...
1,TRX,index_abs,ValueError: Fewer than 3 usable folds for dele...


## 3. Development folds: block increments on common rows

RMSE ratio below 1 means the block beats persistence on the same rows. The fold-to-fold spread is the uncertainty that matters with this few sessions.

In [4]:
if not folds_df.empty:
    f = folds_df[(folds_df.stage == "fold") & folds_df.fitted]
    summ = (f.groupby(["underlying", "target", "population", "model"])["rmse_ratio_vs_persistence"]
              .agg(["mean", "std", "count"]).round(3))
    display(summ.xs("common", level="population") if "common" in f.population.values else summ)
    print("Admitted features per asset (development coverage >= 80%):")
    display(admission_df.pivot_table(index="feature", columns="underlying",
                                     values="admitted", aggfunc="first"))

mean    std  count
underlying target    model                                    
AVAX       index_abs B0_returns_vol_clock  0.747  0.024      7
                     B1_flow_basis         0.750  0.024      7
                     B2_l2_book            0.754  0.026      7
                     B3_atm_options        0.764  0.032      7
           parkinson B0_returns_vol_clock  0.807  0.032      7
                     B1_flow_basis         0.802  0.026      7
                     B2_l2_book            0.810  0.036      7
                     B3_atm_options        0.815  0.034      7
BTC        index_abs B0_returns_vol_clock  0.729  0.032      7
                     B1_flow_basis         0.733  0.032      7
                     B2_l2_book            0.737  0.042      7
                     B3_atm_options        0.730  0.035      7
           parkinson B0_returns_vol_clock  0.847  0.040      7
                     B1_flow_basis         0.856  0.052      7
                     B2_l2_book            0.850  0.055      7
                     B3_atm_options        0.836  0.047      7
ETH        index_abs B0_returns_vol_clock  0.720  0.029      7
                     B1_flow_basis         0.723  0.026      7
                     B2_l2_book            0.726  0.029      7
                     B3_atm_options        0.719  0.024      7
           parkinson B0_returns_vol_clock  0.814  0.023      7
                     B1_flow_basis         0.819  0.032      7
                     B2_l2_book            0.826  0.038      7
                     B3_atm_options        0.815  0.033      7
HYPE       index_abs B0_returns_vol_clock  0.757  0.030      7
                     B1_flow_basis         0.779  0.025      6
                     B2_l2_book            0.796  0.027      4
                     B3_atm_options        0.804  0.036      4
           parkinson B0_returns_vol_clock  0.884  0.091      7
                     B1_flow_basis         0.914  0.069      6
                     B2_l2_book            0.931  0.111      4
                     B3_atm_options        0.918  0.105      4
SOL        index_abs B0_returns_vol_clock  0.740  0.026      7
                     B1_flow_basis         0.746  0.028      7
                     B2_l2_book            0.750  0.037      7
                     B3_atm_options        0.754  0.038      7
           parkinson B0_returns_vol_clock  0.836  0.049      7
                     B1_flow_basis         0.838  0.051      7
                     B2_l2_book            0.839  0.047      7
                     B3_atm_options        0.835  0.044      7
TRX        index_abs B0_returns_vol_clock  0.665  0.082      4
           parkinson B0_returns_vol_clock  0.795  0.124      4
XRP        index_abs B0_returns_vol_clock  0.745  0.052      7
                     B1_flow_basis         0.751  0.045      7
                     B2_l2_book            0.755  0.041      7
                     B3_atm_options        0.754  0.035      7
           parkinson B0_returns_vol_clock  0.766  0.054      7
                     B1_flow_basis         0.756  0.065      7
                     B2_l2_book            0.749  0.064      7
                     B3_atm_options        0.749  0.069      7

Admitted features per asset (development coverage >= 80%):


underlying,AVAX,BTC,ETH,HYPE,SOL,TRX,XRP
feature,,,,,,,
atm_iv_d15,True,True,True,True,True,False,True
atm_iv_d30,True,True,True,True,True,False,True
atm_iv_d5,True,True,True,True,True,False,True
buy_share,True,True,True,True,True,True,True
clock_cos1,True,True,True,True,True,True,True
clock_cos2,True,True,True,True,True,True,True
clock_sin1,True,True,True,True,True,True,True
clock_sin2,True,True,True,True,True,True,True
close_vs_vwap_bps,True,True,True,True,True,True,True


## 4. Final test (scored once)

In [5]:
if not folds_df.empty:
    t = folds_df[(folds_df.stage == "test") & folds_df.fitted]
    display(t.pivot_table(index=["underlying", "target", "population"], columns="model",
                          values=["r2_vs_train_mean", "rmse_ratio_vs_persistence"]).round(3))
    display(t.pivot_table(index=["underlying", "target", "population"], columns="model",
                          values="n_score"))
display(reduced_df.round(3))

r2_vs_train_mean                           \
model                           B0_returns_vol_clock B1_flow_basis B2_l2_book   
underlying target    population                                                 
AVAX       index_abs common                    0.092         0.089      0.083   
                     own                       0.094         0.086      0.081   
           parkinson common                    0.283         0.304      0.295   
                     own                       0.306         0.302      0.293   
BTC        index_abs common                    0.062         0.054      0.032   
                     own                       0.065         0.059      0.049   
           parkinson common                    0.407         0.418      0.363   
                     own                       0.405         0.411      0.376   
ETH        index_abs common                    0.121         0.106      0.105   
                     own                       0.119         0.094      0.090   
           parkinson common                    0.487         0.456      0.470   
                     own                       0.483         0.427      0.434   
HYPE       index_abs common                    0.091         0.088      0.095   
                     own                       0.073         0.070      0.075   
           parkinson common                    0.045         0.110      0.105   
                     own                       0.076         0.131      0.107   
SOL        index_abs common                    0.120         0.114      0.098   
                     own                       0.103         0.085      0.086   
           parkinson common                    0.469         0.448      0.472   
                     own                       0.447         0.397      0.433   
TRX        index_abs common                    0.158           NaN        NaN   
                     own                       0.164           NaN        NaN   
           parkinson common                    0.041           NaN        NaN   
                     own                       0.041           NaN        NaN   
XRP        index_abs common                    0.231         0.226      0.228   
                     own                       0.207         0.190      0.193   
           parkinson common                    0.287         0.297      0.279   
                     own                       0.274         0.259      0.251   

                                               rmse_ratio_vs_persistence  \
model                           B3_atm_options      B0_returns_vol_clock   
underlying target    population                                            
AVAX       index_abs common              0.081                     0.719   
                     own                 0.081                     0.719   
           parkinson common              0.277                     0.796   
                     own                 0.277                     0.792   
BTC        index_abs common              0.033                     0.718   
                     own                 0.033                     0.718   
           parkinson common              0.362                     0.853   
                     own                 0.362                     0.850   
ETH        index_abs common              0.098                     0.723   
                     own                 0.098                     0.724   
           parkinson common              0.481                     0.870   
                     own                 0.481                     0.865   
HYPE       index_abs common              0.095                     0.745   
                     own                 0.095                     0.742   
           parkinson common              0.084                     0.811   
                     own                 0.084                     0.796   
SOL        index_abs common              0.097                     0.731   
    

model                            B0_returns_vol_clock  B1_flow_basis  \
underlying target    population                                        
AVAX       index_abs common                     660.0          660.0   
                     own                        756.0          756.0   
           parkinson common                     660.0          660.0   
                     own                        756.0          756.0   
BTC        index_abs common                     811.0          811.0   
                     own                        883.0          883.0   
           parkinson common                     811.0          811.0   
                     own                        883.0          883.0   
ETH        index_abs common                     794.0          794.0   
                     own                        885.0          885.0   
           parkinson common                     794.0          794.0   
                     own                        885.0          885.0   
HYPE       index_abs common                     692.0          692.0   
                     own                        778.0          778.0   
           parkinson common                     692.0          692.0   
                     own                        778.0          778.0   
SOL        index_abs common                     670.0          670.0   
                     own                        885.0          885.0   
           parkinson common                     670.0          670.0   
                     own                        885.0          885.0   
TRX        index_abs common                      47.0            NaN   
                     own                         47.0            NaN   
           parkinson common                      47.0            NaN   
                     own                         47.0            NaN   
XRP        index_abs common                     639.0          639.0   
                     own                        841.0          841.0   
           parkinson common                     639.0          639.0   
                     own                        841.0          841.0   

model                            B2_l2_book  B3_atm_options  
underlying target    population                              
AVAX       index_abs common           660.0           660.0  
                     own              756.0           660.0  
           parkinson common           660.0           660.0  
                     own              756.0           660.0  
BTC        index_abs common           811.0           811.0  
                     own              883.0           811.0  
           parkinson common           811.0           811.0  
                     own              883.0           811.0  
ETH        index_abs common           794.0           794.0  
                     own              885.0           794.0  
           parkinson common           794.0           794.0  
                     own              885.0           794.0  
HYPE       index_abs common           692.0           692.0  
                     own              778.0           692.0  
           parkinson common           692.0           692.0  
                     own              778.0           692.0  
SOL        index_abs common           670.0           670.0  
                     own              885.0           670.0  
           parkinson common           670.0           670.0  
                     own              885.0           670.0  
TRX        index_abs common             NaN             NaN  
                     own                NaN             NaN  
           parkinson common             NaN             NaN  
                     own                NaN             NaN  
XRP        index_abs common           639.0           639.0  
                     own              841.0           639.0  
           parkinson common           639.0           639.0  
                     own              841

,underlying,target,model,p,n_train,n_test,fitted,reason,log_rmse,r2_vs_train_mean,persistence_log_rmse,rmse_ratio_vs_persistence
0,AVAX,parkinson,B2_l2_book_start,38,2393,756,True,,1.296,0.293,1.629,0.795
1,AVAX,parkinson,B2_l2_book_reduced,24,2393,756,True,,1.286,0.304,1.629,0.789
2,AVAX,index_abs,B2_l2_book_start,38,2393,756,True,,1.032,0.081,1.428,0.723
3,AVAX,index_abs,B2_l2_book_reduced,24,2393,756,True,,1.037,0.073,1.428,0.726
4,BTC,parkinson,B2_l2_book_start,38,2745,883,True,,0.480,0.376,0.555,0.866
5,BTC,parkinson,B2_l2_book_reduced,25,2745,883,True,,0.478,0.381,0.555,0.862
6,BTC,index_abs,B2_l2_book_start,38,2745,883,True,,1.126,0.049,1.555,0.724
7,BTC,index_abs,B2_l2_book_reduced,23,2745,883,True,,1.122,0.055,1.555,0.722
8,ETH,parkinson,B2_l2_book_start,38,2746,885,True,,0.480,0.434,0.540,0.888
9,ETH,parkinson,B2_l2_book_reduced,28,2746,885,True,,0.480,0.434,0.540,0.888


## 5. Deletion path, coverage by hour, marginal bottlenecks, residuals by collector vintage

In [6]:
if not deletion_df.empty:
    display(deletion_df)
if not clock_df.empty:
    display(clock_df.pivot_table(index="utc_hour", columns=["model"], values="share",
                                 aggfunc="median").round(2))
if not loss_df.empty:
    display(loss_df.groupby("feature")["unique_loss_rows"].median().sort_values(ascending=False).head(10))
if not vintage_df.empty:
    display(vintage_df[vintage_df.population == "own"].pivot_table(
        index=["underlying", "target", "vintage"], columns="model", values="mean_residual").round(3))

,underlying,target,step,removed,p_before,mean_fold_gain,folds_improved,folds
0,AVAX,parkinson,1,weekend,39,0.012261,5,7
1,AVAX,parkinson,2,clock_sin1,38,0.007170,5,7
2,AVAX,parkinson,3,obi_20_change,37,0.001557,5,7
3,AVAX,parkinson,4,log_avg_trade,36,0.001383,4,7
4,AVAX,parkinson,5,ret_30m,35,0.000956,5,7
...,...,...,...,...,...,...,...,...
160,XRP,index_abs,11,perp_mark_basis_bps,29,0.000381,4,7
161,XRP,index_abs,12,log_range_bps,28,0.000297,4,7
162,XRP,index_abs,13,oi_change,27,0.000288,4,7
163,XRP,index_abs,14,signed_volume,26,0.000267,4,7


model,B0_returns_vol_clock,B1_flow_basis,B2_l2_book,B3_atm_options
utc_hour,,,,
0,0.79,0.73,0.73,0.68
1,0.93,0.87,0.87,0.78
2,0.98,0.92,0.92,0.85
3,0.91,0.85,0.85,0.72
4,1.00,0.93,0.93,0.76
5,0.95,0.89,0.89,0.64
6,0.87,0.81,0.81,0.35
7,0.77,0.77,0.77,0.02
8,0.98,0.93,0.93,0.38


feature
log_rv_1h            93.0
log_rv_4h_tol        84.0
atm_iv_d30           47.0
atm_iv_d15           15.0
atm_iv_d5            13.5
buy_share             0.0
clock_cos1            0.0
clock_sin2            0.0
close_vs_vwap_bps     0.0
clock_cos2            0.0
Name: unique_loss_rows, dtype: float64

model                                         B0_returns_vol_clock  \
underlying target    vintage                                         
AVAX       index_abs eventtime_fix_unstamped                -0.235   
                     futures_5m_l2_v0.3.0                   -0.180   
                     futures_5m_l2_v0.3.1                   -0.017   
           parkinson eventtime_fix_unstamped                -0.616   
                     futures_5m_l2_v0.3.0                   -0.110   
                     futures_5m_l2_v0.3.1                    0.295   
BTC        index_abs eventtime_fix_unstamped                 0.062   
                     futures_5m_l2_v0.3.0                    0.023   
                     futures_5m_l2_v0.3.1                   -0.034   
           parkinson eventtime_fix_unstamped                -0.014   
                     futures_5m_l2_v0.3.0                    0.049   
                     futures_5m_l2_v0.3.1                   -0.069   
ETH        index_abs eventtime_fix_unstamped                -0.106   
                     futures_5m_l2_v0.3.0                    0.033   
                     futures_5m_l2_v0.3.1                   -0.061   
           parkinson eventtime_fix_unstamped                -0.039   
                     futures_5m_l2_v0.3.0                    0.050   
                     futures_5m_l2_v0.3.1                   -0.082   
HYPE       index_abs eventtime_fix_unstamped                -0.092   
                     futures_5m_l2_v0.3.0                    0.034   
                     futures_5m_l2_v0.3.1                   -0.188   
           parkinson eventtime_fix_unstamped                 0.079   
                     futures_5m_l2_v0.3.0                    0.536   
                     futures_5m_l2_v0.3.1                    0.339   
SOL        index_abs eventtime_fix_unstamped                -0.064   
                     futures_5m_l2_v0.3.0                   -0.066   
                     futures_5m_l2_v0.3.1                   -0.126   
           parkinson eventtime_fix_unstamped                -0.037   
                     futures_5m_l2_v0.3.0                   -0.034   
                     futures_5m_l2_v0.3.1                   -0.135   
TRX        index_abs futures_5m_l2_v0.3.0                   -0.096   
                     futures_5m_l2_v0.3.1                    0.198   
           parkinson futures_5m_l2_v0.3.0                    0.554   
                     futures_5m_l2_v0.3.1                   -0.315   
XRP        index_abs eventtime_fix_unstamped                -0.159   
                     futures_5m_l2_v0.3.0                   -0.171   
                     futures_5m_l2_v0.3.1                   -0.188   
           parkinson eventtime_fix_unstamped                 0.060   
                     futures_5m_l2_v0.3.0                    0.336   
                     futures_5m_l2_v0.3.1                    0.095   

model                                         B1_flow_basis  B2_l2_book  \
underlying target    vintage                                              
AVAX       index_abs eventtime_fix_unstamped         -0.287      -0.277   
                     futures_5m_l2_v0.3.0            -0.194      -0.206   
                     futures_5m_l2_v0.3.1            -0.017      -0.027   
           parkinson eventtime_fix_unstamped         -0.627      -0.593   
                     futures_5m_l2_v0.3.0            -0.068       0.050   
                     futures_5m_l2_v0.3.1             0.306       0.357   
BTC        index_abs eventtime_fix_unstamped          0.033       0.064   
                     futures_5m_l2_v0.3.0            -0.020      -0.022   
                     futures_5m_l2_v0.3.1            -0.053      -0.058   
           parkinson eventtime_fix_unstamped         -0.028       0.006   
                     futures_5m_l2_v0.3.0             0.016      -0.015   
                     futures_5m_l2_v0.3.1            -0.063      -0.109   
ETH      

## 6. Save outputs (no database writes)

In [7]:
outputs = {
    "03_plan.csv": plan_df,
    "03_feature_admission.csv": admission_df,
    "03_block_scores.csv": folds_df,
    "03_reduced_test.csv": reduced_df,
    "03_deletion_history.csv": deletion_df,
    "03_clock_coverage.csv": clock_df,
    "03_unique_loss.csv": loss_df,
    "03_test_residual_by_vintage.csv": vintage_df,
    "03_fit_errors.csv": errors_df,
}
for name, table in outputs.items():
    table.to_csv(OUTPUT / name, index=False)
    print(name, "rows:", len(table))
summary = {
    "version": vb.VOL_OLS_BLOCKS_VERSION,
    "base_version": base.VOL_OLS_5M_VERSION,
    "targets": list(base.TARGETS),
    "blocks": {b: vb.BLOCKS[b] for b in vb.BLOCK_ORDER},
    "excluded_by_design": list(vb.EXCLUDED_BY_DESIGN),
    "feature_min_coverage_dev": vb.FEATURE_MIN_COVERAGE,
    "min_rows_per_param": vb.MIN_ROWS_PER_PARAM,
    "test_fraction": TEST_FRACTION, "min_train_sessions": MIN_TRAIN_SESSIONS,
    "delete_block": DELETE_BLOCK, "max_drops": MAX_DROPS,
    "data_span": [str(df.state_time.min()), str(df.state_time.max())],
    "vintage_boundaries_file": vintage_path.name if boundaries is not None else None,
    "failed": errors,
}
(OUTPUT / "03_run_metadata.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
print("Done. Commit the notebook and 03_* outputs.")

03_plan.csv rows: 7
03_feature_admission.csv rows: 301
03_block_scores.csv rows: 896
03_reduced_test.csv rows: 24
03_deletion_history.csv rows: 165
03_clock_coverage.csv rows: 672
03_unique_loss.csv rows: 297
03_test_residual_by_vintage.csv rows: 296
03_fit_errors.csv rows: 2
Done. Commit the notebook and 03_* outputs.


## Interpretation contract

1. A block's value is its increment over the previous block on **common** rows, read across folds, not one test number.
2. A feature that is not admitted, or a block that fails the parameter budget, has no evidence either way.
3. A block that helps under `parkinson` but not under `index_abs` (e.g. flow) may be predicting the trade-range measurement rather than volatility.
4. If B2 residuals differ systematically by collector vintage, L2 coefficients are not stable across the event-time fix and must not be interpreted.
5. Research diagnostics only.